# 🏠 Housing Data Analysis and K-Means Clustering

## 📌 Overview
This notebook demonstrates basic Exploratory Data Analysis (EDA) and **K-Means Clustering** on a housing dataset.
The goal is to analyze housing features and group similar houses into clusters based on their characteristics.

## 🎯 Objectives
- Load and explore the housing dataset.
- Visualize housing data using plots.
- Preprocess numerical features.
- Apply K-Means Clustering.
- Visualize the resulting clusters.

## 📂 Dataset
The dataset used is **Housing.csv**, containing information about houses such as Area, Bedrooms, Price, and other housing-related attributes.


In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Modeling
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

%matplotlib inline


## 1. Load Dataset

If you have your own `Housing.csv`, upload it and uncomment the `pd.read_csv` line below.
Otherwise, a synthetic dataset with the **same schema** is generated so this notebook runs end-to-end on its own.

In [ ]:
# To use your own data, upload Housing.csv and run:
# df = pd.read_csv("Housing.csv")

# --- Synthetic dataset generator (same schema as the classic Housing dataset) ---
np.random.seed(42)
n = 545  # matches typical size of the Kaggle Housing dataset

area = np.random.randint(1500, 16000, n)
bedrooms = np.random.choice([1, 2, 3, 4, 5, 6], n, p=[0.03, 0.15, 0.35, 0.30, 0.12, 0.05])
bathrooms = np.random.choice([1, 2, 3, 4], n, p=[0.5, 0.35, 0.10, 0.05])
stories = np.random.choice([1, 2, 3, 4], n, p=[0.4, 0.35, 0.20, 0.05])
parking = np.random.choice([0, 1, 2, 3], n, p=[0.3, 0.35, 0.25, 0.10])
mainroad = np.random.choice(["yes", "no"], n, p=[0.85, 0.15])
guestroom = np.random.choice(["yes", "no"], n, p=[0.3, 0.7])
basement = np.random.choice(["yes", "no"], n, p=[0.35, 0.65])
hotwaterheating = np.random.choice(["yes", "no"], n, p=[0.1, 0.9])
airconditioning = np.random.choice(["yes", "no"], n, p=[0.4, 0.6])
prefarea = np.random.choice(["yes", "no"], n, p=[0.25, 0.75])
furnishingstatus = np.random.choice(["furnished", "semi-furnished", "unfurnished"], n, p=[0.3, 0.4, 0.3])

price = (
    area * 250
    + bedrooms * 150000
    + bathrooms * 300000
    + stories * 200000
    + parking * 100000
    + (mainroad == "yes") * 400000
    + (airconditioning == "yes") * 350000
    + (prefarea == "yes") * 300000
    + (basement == "yes") * 150000
    + np.random.normal(0, 400000, n)
)
price = np.clip(price, 1500000, None).round(-3)

df = pd.DataFrame({
    "price": price,
    "area": area,
    "bedrooms": bedrooms,
    "bathrooms": bathrooms,
    "stories": stories,
    "mainroad": mainroad,
    "guestroom": guestroom,
    "basement": basement,
    "hotwaterheating": hotwaterheating,
    "airconditioning": airconditioning,
    "parking": parking,
    "prefarea": prefarea,
    "furnishingstatus": furnishingstatus,
})

df.head()


## 2. Explore the Dataset

In [ ]:
print("Shape:", df.shape)
df.info()


In [ ]:
df.describe()


In [ ]:
# Check for missing values
df.isnull().sum()


## 3. Data Visualization

### 3.1 Histogram — Distribution of House Prices

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["price"], bins=30, color="steelblue", edgecolor="black")
plt.title("Distribution of House Prices")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()


### 3.2 Scatter Plot — Area vs Price

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["area"], df["price"], alpha=0.6, color="darkorange")
plt.title("Area vs Price")
plt.xlabel("Area")
plt.ylabel("Price")
plt.show()


## 4. Preprocess Numerical Features

Select numerical features for clustering: **Area, Bedrooms, Price**.

In [ ]:
features = df[["area", "bedrooms", "price"]].copy()

# Handle missing values (none expected here, but kept for robustness)
features = features.dropna()

# Standardize the data
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

scaled_features[:5]


## 5. Choosing the Number of Clusters

Using the elbow method to see why **k = 3** is a reasonable choice.

In [ ]:
inertia = []
k_range = range(1, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(scaled_features)
    inertia.append(km.inertia_)

plt.figure(figsize=(7, 5))
plt.plot(list(k_range), inertia, marker="o")
plt.title("Elbow Method for Optimal k")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.show()


## 6. Apply K-Means Clustering

Applying K-Means with **3 clusters**, as in the original project.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(scaled_features)

df[["area", "bedrooms", "price", "cluster"]].head(10)


In [ ]:
df["cluster"].value_counts().sort_index()


## 7. Visualize the Clusters

In [ ]:
plt.figure(figsize=(8, 6))
colors = ["#4C72B0", "#DD8452", "#55A868"]

for cluster_id in sorted(df["cluster"].unique()):
    subset = df[df["cluster"] == cluster_id]
    plt.scatter(subset["area"], subset["price"], label=f"Cluster {cluster_id}",
                color=colors[cluster_id], alpha=0.7)

plt.title("House Clusters by Area and Price (K-Means, k=3)")
plt.xlabel("Area")
plt.ylabel("Price")
plt.legend()
plt.show()


In [ ]:
# Cluster centers (in original feature scale)
centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

centers_df = pd.DataFrame(centers_original, columns=["area", "bedrooms", "price"])
centers_df.index.name = "cluster"
centers_df


In [ ]:
# Average feature values per cluster — helps interpret what each cluster represents
df.groupby("cluster")[["area", "bedrooms", "price"]].mean().round(0)


## 8. Results

The houses were successfully grouped into **3 clusters** based on area, bedrooms, and price. Looking at the cluster centers helps interpret each group, for example:

- **Cluster with smaller area & lower price** → budget/compact houses
- **Cluster with mid-range area & price** → typical mid-market houses
- **Cluster with larger area & higher price** → premium/luxury houses

The clustering visualization helps identify patterns among different types of houses and their pricing characteristics.

## 📚 Learning Outcomes
- Understanding of Exploratory Data Analysis (EDA).
- Experience with data visualization using Matplotlib.
- Knowledge of feature scaling.
- Practical implementation of K-Means Clustering.
- Interpretation of clustering results in real-world datasets.

**Note:** This notebook ships with a synthetic dataset matching the original schema, so every cell runs immediately. Swap in your real `Housing.csv` in the "Load Dataset" cell for results on actual data.
